In [3]:
import pandas as pd
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report


IN_PATH = r"C:\Users\salma\Documents\GitHub\Machine-Learning\data\processed\H2_clean.csv"
OUT_PATH = r"C:\Users\salma\Documents\GitHub\Machine-Learning\src\hotel_booking\classification\classification6Col.csv"

features = [
    "MarketSegment",
    "TotalOfSpecialRequests",
    "Country",
    "LeadTime",
    "CustomerType",
    "arrival_days",
    "RequiredCarParkingSpaces",
]
target = "IsCanceled"

df = pd.read_csv(IN_PATH)

missing = [c for c in features + [target] if c not in df.columns]
assert not missing, f"Columns not found in the clean CSV: {missing}"

df6 = df[features + [target]]

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df6.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH, df6.shape)
print(df6.dtypes)
print(df6.isnull().sum().sum(), "missing values")
print(df6[target].value_counts(normalize=True).round(3))





Saved: C:\Users\salma\Documents\GitHub\Machine-Learning\src\hotel_booking\classification\classification6Col.csv (52026, 8)
MarketSegment                 str
TotalOfSpecialRequests      int64
Country                       str
LeadTime                    int64
CustomerType                  str
arrival_days                int64
RequiredCarParkingSpaces    int64
IsCanceled                  int64
dtype: object
0 missing values
IsCanceled
0    0.696
1    0.304
Name: proportion, dtype: float64


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# load + split
df = pd.read_csv(r"C:\Users\salma\Documents\GitHub\Machine-Learning\src\hotel_booking\classification\classification6Col.csv")
y = df["IsCanceled"]
X = df.drop(columns=["IsCanceled"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# no scaling needed for trees; one-hot the text columns
prep = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

model = Pipeline([
    ("prep", prep),
    ("clf", HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=500, max_leaf_nodes=63,
        l2_regularization=1.0, early_stopping=True, random_state=42)),
])

model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Baseline (always 'not cancelled'):", round(1 - y_test.mean(), 4))
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("ROC-AUC :", round(roc_auc_score(y_test, proba), 4))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

Baseline (always 'not cancelled'): 0.6959
Accuracy: 0.7916
ROC-AUC : 0.854
[[6447  795]
 [1374 1790]]
              precision    recall  f1-score   support

           0      0.824     0.890     0.856      7242
           1      0.692     0.566     0.623      3164

    accuracy                          0.792     10406
   macro avg      0.758     0.728     0.739     10406
weighted avg      0.784     0.792     0.785     10406

